# Домашнее задание: Проектирование ИИ-агента на базе LLM

В этом домашнем задании вы пройдете путь от создания базового агента с кастомными инструментами до разработки защищенной мультиагентной системы с человеком в контуре (human-in-the-loop).

**Важное напоминание:** В рамках этого ДЗ вы можете использовать **любые технологии и фреймворки** для реализации задач. Однако мы настоятельно рекомендуем использовать **LangChain** для стандартной части и **LangGraph** для продвинутой - они дают удобные абстракции и хорошо документированы.

**Рекомендация по LLM:** Для отладки агентов со сложной логикой вызова инструментов рекомендуем начинать с больших моделей через [OpenRouter](https://openrouter.ai/) или любой другой сервис (к примеру гигачат, яндекс облако).
---

## Структура ДЗ (100 баллов)

| Часть | Подзадание | Баллы |
|---|---|---|
| Стандартная | 1.1 - 1.3 Реализация 3 инструментов | 20 |
| Стандартная | 1.4 Промпт-инженерия и создание ReAct агента | 10 |
| Стандартная | 1.5 Тестирование базового агента | 10 |
| Стандартная | 1.6 Анализ рисков и идеи по улучшению | 10 |
| Продвинутая | 2.1 Переход на LangGraph | 10 |
| Продвинутая | 2.2 - 2.3 Оркестратор и субагенты | 15 |
| Продвинутая | 2.4 Human-in-the-loop | 10 |
| Продвинутая | 2.5 Финальное тестирование | 5 |
| Продвинутая | 2.6 Анализ рисков и идеи по улучшению | 10 |


---
## Установка зависимостей

Установите необходимые библиотеки. Если вы выбрали инструменты, требующие дополнительных пакетов (например, `yfinance` для курсов валют или `feedparser` для новостей), добавьте их сюда.


In [3]:
!pip install -qU langchain langchain-openai langgraph datasets matplotlib pandas requests feedparser yfinance

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.1/144.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.8 MB/s eta 0:00:00

In [4]:
import os

os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
     model="anthropic/claude-3.5-sonnet",
     openai_api_base="https://openrouter.ai/api/v1",
     openai_api_key=os.environ["OPENAI_API_KEY"]
)


---
# Часть 1. Стандартная (50 баллов)

В этой части вам нужно:
1. Выбрать и реализовать три инструмента из предложенного списка.
2. Написать системный промпт и создать ReAct агента.
3. Протестировать агента на разных запросах.
4. Проанализировать риски и предложить идеи по улучшению.


### 1.1 - 1.3 Реализация инструментов (20 баллов)

Выберите **три любых инструмента** из списка ниже и реализуйте их:

1. **Поиск по базе знаний** - загрузите датасет `data-silence/rus_news_classifier` с HuggingFace (около 70k коротких русских новостей, поля: `news` - текст, `labels` - категория). Реализуйте поиск по ключевым словам или TF-IDF.
2. **Калькулятор сложных процентов** - функция принимает начальную сумму, годовую ставку (%), срок в годах и частоту капитализации в год.
3. **Построение графиков** - принимает данные (или путь к CSV), строит график через Matplotlib, сохраняет в файл и возвращает путь к нему.
4. **Текущий курс валют** - через публичный API ЦБ РФ (`https://cbr.ru/scripts/XML_daily.asp`, без ключа) или через `yfinance`.
5. **Текущая погода** - через `wttr.in` (без ключа, например: `requests.get("https://wttr.in/Москва?format=j1")`).
6. **Последние новости** - парсинг RSS-ленты любого СМИ через `feedparser` (например, `https://lenta.ru/rss/news`).

**Подсказки по реализации инструментов в LangChain:**
- Используйте декоратор `@tool` из `langchain_core.tools`.
- Пишите подробные docstring - именно по ним LLM понимает, когда и как вызывать инструмент.
- Указывайте типы аргументов (type hints) - это помогает LLM правильно формировать вызов.
- Инструмент должен возвращать строку или что-то, что легко преобразуется в строку.
- Обрабатывайте исключения внутри инструмента и возвращайте понятное сообщение об ошибке.

Пример структуры инструмента:
```python
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Возвращает текущую погоду в указанном городе.
    Используй этот инструмент, когда пользователь спрашивает о погоде.
    Args:
        city: Название города на русском или английском языке.
    """
    try:
        # ваша реализация
        pass
    except Exception as e:
        return f"Ошибка при получении погоды: {e}"
```


In [2]:
# Инструмент 1
from langchain_core.tools import tool
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

_news_df = None
_tfidf_matrix = None
_vectorizer = None

def _load_news_dataset():
    global _news_df, _tfidf_matrix, _vectorizer
    if _news_df is not None:
        return
    dataset = load_dataset("data-silence/rus_news_classifier", split="train")
    data = dataset.select(range(min(10000, len(dataset))))
    _news_df = pd.DataFrame({"text": data["news"], "label": data["labels"]})
    _vectorizer = TfidfVectorizer(max_features=5000, stop_words=None)
    _tfidf_matrix = _vectorizer.fit_transform(_news_df["text"])

@tool
def search_news(query: str, top_k: int = 3) -> str:
    """Поиск новостей по ключевой фразе. query – строка, top_k – целое (по умолчанию 3)."""
    try:
        _load_news_dataset()
        q_vec = _vectorizer.transform([query])
        scores = cosine_similarity(q_vec, _tfidf_matrix).flatten()
        top_indices = scores.argsort()[-top_k:][::-1]
        results = []
        for idx in top_indices:
            text = _news_df.iloc[idx]["text"]
            if len(text) > 200:
                text = text[:200] + "..."
            results.append(f"[{_news_df.iloc[idx]['label']}] {text}")
        return "\n\n".join(results) if results else "Ничего не найдено."
    except Exception as e:
        return f"Ошибка: {e}"

# Инструмент 2
@tool
def compound_interest(principal: float, annual_rate: float, years: int, compounding_per_year: int = 12) -> str:
    """Расчёт процентов. principal, annual_rate, years, compounding_per_year (по умолчанию 12)."""
    try:
        rate = annual_rate / 100.0
        total = principal * (1 + rate / compounding_per_year) ** (compounding_per_year * years)
        profit = total - principal
        return (f"Итоговая сумма: {total:.2f}\nПрибыль: {profit:.2f}")
    except Exception as e:
        return f"Ошибка: {e}"

# Инструмент 3
import requests
import xml.etree.ElementTree as ET
from datetime import datetime

@tool
def get_exchange_rate(currency_code: str = "USD") -> str:
    """Курс валюты от ЦБ РФ. currency_code – трёхбуквенный код."""
    try:
        url = "https://cbr.ru/scripts/XML_daily.asp"
        resp = requests.get(url, timeout=10)
        resp.encoding = "windows-1251"
        if resp.status_code != 200:
            return "Не удалось получить данные от ЦБ РФ."
        root = ET.fromstring(resp.text)
        date_str = root.attrib.get("Date", "")
        date_obj = datetime.strptime(date_str, "%d.%m.%Y")
        date_formatted = date_obj.strftime("%d.%m.%Y")
        for valute in root.findall("Valute"):
            if valute.find("CharCode").text == currency_code.upper():
                nominal = int(valute.find("Nominal").text)
                value = float(valute.find("Value").text.replace(",", "."))
                rate = value / nominal
                return f"Курс {currency_code.upper()} на {date_formatted}: {rate:.4f} руб."
        return f"Валюта {currency_code} не найдена."
    except Exception as e:
        return f"Ошибка: {e}"

### 1.4 Промпт-инженерия и создание ReAct агента (10 баллов)

**Задание:**
1. Напишите системный промпт для агента. Задайте ему персону (например, "опытный финансовый консультант" или "строгий корпоративный помощник").
2. Промпт должен явно запрещать агенту отвечать на вопросы, выходящие за рамки его инструментов - это защита от галлюцинаций.
3. Создайте ReAct агента с помощью LangChain и подключите к нему ваши инструменты.

**Подсказки:**
- В LangChain используйте `create_react_agent` из `langchain.agents` и `AgentExecutor`.
- Передайте системный промпт через `ChatPromptTemplate` или параметр `agent_kwargs`.
- Установите `verbose=True` в `AgentExecutor` - так вы будете видеть все промежуточные шаги (мысли агента, вызовы инструментов, ответы инструментов). Это очень полезно для отладки.
- Установите `handle_parsing_errors=True` - это защитит от падений при некорректном ответе LLM.
- Параметр `max_iterations` ограничивает количество шагов агента и защищает от бесконечных циклов.

Пример создания агента:
```python
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="anthropic/claude-3.5-sonnet", ...)
tools = [tool_1, tool_2, tool_3]

# Можно взять готовый промпт из hub или написать свой
prompt = hub.pull("hwchase17/react")

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)
```


In [3]:
import os
from langchain.agents import initialize_agent, AgentType
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = "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"

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
    max_tokens=1500
)

tools = [search_news, compound_interest, get_exchange_rate]

prefix = """
Ты — опытный финансовый консультант. Используй только предоставленные инструменты.

Строгие правила:
- Отвечай ТОЛЬКО на вопросы, которые можно решить с помощью инструментов.
- Если вопрос вне компетенции – вежливо откажись.
- НИКОГДА не выдумывай числа – только результаты инструментов.
- Отвечай на русском языке.
"""

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,
    agent_kwargs={"prefix": prefix}
)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


### 1.5 Тестирование базового агента (10 баллов)

Протестируйте вашего агента на различных запросах. Покажите вывод промежуточных шагов.

Задайте минимум 3 запроса:
1. Запрос, требующий вызова только одного инструмента.
2. Сложный запрос, требующий вызова двух инструментов последовательно.
3. Провокационный запрос вне компетенции агента (проверка защиты от галлюцинаций).

**Подсказка:** Используйте `agent_executor.invoke({"input": "ваш запрос"})`. Вывод `verbose=True` покажет все шаги рассуждений.


In [37]:
print("Один инструмент")
result1 = agent.invoke({"input": "Какой сегодня курс доллара?"})
print("\nОтвет:\n", result1["output"])

Один инструмент


> Entering new AgentExecutor chain...
Action:
```
{
  "action": "get_exchange_rate",
  "action_input": {"currency_code": "USD"}
}
```

Observation: Курс USD на 28.07.2026: 78.0172 руб.
Thought:Action:
```
{
  "action": "get_exchange_rate",
  "action_input": {"currency_code": "USD"}
}
```

Observation: Курс USD на 28.07.2026: 78.0172 руб.
Thought:Action:
```
{
  "action": "get_exchange_rate",
  "action_input": {"currency_code": "USD"}
}
```

Observation: Курс USD на 28.07.2026: 78.0172 руб.
Thought:Action:
```
{
  "action": "get_exchange_rate",
  "action_input": {"currency_code": "USD"}
}
```

Observation: Курс USD на 28.07.2026: 78.0172 руб.
Thought:Action:
```
{
  "action": "get_exchange_rate",
  "action_input": {"currency_code": "USD"}
}
```

Observation: Курс USD на 28.07.2026: 78.0172 руб.
Thought:

> Finished chain.

Ответ:
 Agent stopped due to iteration limit or time limit.


In [38]:
print("Два инструмента")
result2 = agent.invoke({"input": "Расскажи, что происходит в экономике России, и покажи текущий курс евро."})
print("\nОтвет:\n", result2["output"])


Два инструмента


> Entering new AgentExecutor chain...
Thought: Сначала я найду новости о текущей экономической ситуации в России, а затем проверю курс евро. 

Action:
```
{
  "action": "search_news",
  "action_input": {"query": "экономика России", "top_k": 3}
}
```

Observation: [3] Европейская экономика сейчас испытывает более серьезные сложности, чем американская, что отражается на курсе валют, заявил кандидат экономических наук, финансовый аналитик Михаил Беляев. Так в разгово...

[3] Быстрое восстановление российской экономики зависит от снятия ограничительных мер западных стран, которые были введены после начала спецоперации на Украине. Без этого условия на реабилитацию уйдет не ...

[4] латвия рискует окончательно потерять транзит из россии предприниматель рассказал о перспективах латвийского перевалочного бизнеса в своем facebook . он обратил внимание на тот факт , что сейчас стоимо...
Thought:Теперь я найду текущий курс евро. 

Action:
```
{
  "action": "get_exchange_rate",
 

In [39]:
print("вне компетенции")
result3 = agent.invoke({"input": "Какая сегодня погода в Москве?"})
print("\nОтвет:\n", result3["output"])

вне компетенции


> Entering new AgentExecutor chain...
Извините, но я не могу предоставить информацию о погоде.

> Finished chain.

Ответ:
 Извините, но я не могу предоставить информацию о погоде.


### 1.6 Анализ рисков и идеи по улучшению (10 баллов)

Это задание не оценивается в баллах, но является обязательным. Здесь вы должны проявить критическое мышление и осмыслить то, что построили.

**Задание:** Напишите развернутый анализ (минимум 300 слов) в ячейке ниже, ответив на следующие вопросы:

**Риски текущей реализации:**
- Какие ошибки может совершить ваш агент? Приведите конкретные примеры запросов, на которых он может сломаться или дать неверный ответ.
- Что произойдет, если один из внешних API (погода, курсы) будет недоступен? Как агент обработает эту ситуацию?
- Насколько надежен ваш системный промпт? Можно ли обойти его ограничения с помощью хитро сформулированного запроса (prompt injection)?
- Какие риски несет использование больших LLM через внешние API (задержки, стоимость, утечка данных)?

**Гипотезы по улучшению:**
- Как можно улучшить качество поиска в инструменте базы знаний? Что если заменить keyword-поиск на семантический (с эмбеддингами)?
- Как можно сделать агента более устойчивым к ошибкам инструментов? Например, добавить логику повторных попыток или fallback-инструменты.
- Что изменится, если заменить большую LLM на маленькую локальную модель? Какие задачи пострадают в первую очередь?
- Как можно добавить память агенту, чтобы он помнил контекст предыдущих разговоров?

**Идеи по расширению:**
- Какие еще инструменты было бы полезно добавить для вашего конкретного сценария использования?
- Как бы вы оценивали качество работы агента в продакшене? Какие метрики использовали бы?


**Ваш анализ:**

1. Риски текущей реализации

Он может неправильно понять запрос и выбрать не тот инструмент. Например, на вопрос "Какие новости по экономике?" он должен вызвать search_news, но может ошибочно попытаться использовать get_exchange_rate. Есть ещё сложности с передавачей неверных аргументов в инструменты. ("100 тысяч" написать как число 100000) И в тестах я видел, как агент несколько раз вызывал один и тот же инструмент.

Если API ЦБ РФ будет недоступен мой инструмент get_exchange_rate возвращает сообщение об ошибке. В системном промпте я запретил выдумывать данные

К основным рискам я бы отнёс зависимость от провайдера и медлительность(а также возможно слабую модель для реального использования

2. Гипотезы по улучшению

Можно добавить механизм повторных попыток, добавить прошлые запрсоы в промпт и использовать лучшую локальную модель

3. Идеи по расширению:

Возможно добавил бы прогнозы и анализы акций и изменения валют,

Я бы добавил метрики по точности вызова инструментов, время до ответа и удовлетворённость пользователя(с обратной связью)

---
# Часть 2. Продвинутая (50 баллов)

В этой части вы переведете агента на рельсы LangGraph, добавите разделение ролей (Оркестратор и субагенты) и внедрите механизм безопасности (Human-in-the-loop).

**Напоминание:** Вы можете использовать любые технологии. Описанный ниже подход через LangGraph - рекомендация, а не требование.


### 2.1 Переход на LangGraph (10 баллов)

Перепишите базового агента из Части 1 с использованием LangGraph.

**Что нужно сделать:**
1. Определить граф состояния (`StateGraph`) с узлами для LLM и для инструментов.
2. Настроить `conditional_edges` для маршрутизации: если LLM вызвал инструмент - идем в узел инструментов, иначе - завершаем.
3. Скомпилировать граф и визуализировать его.

**Подсказки:**
- Используйте `MessagesState` как базовое состояние - это удобная обертка над списком сообщений.
- Узел агента вызывает LLM с привязанными инструментами: `llm.bind_tools(tools)`.
- Для узла инструментов используйте готовый `ToolNode` из `langgraph.prebuilt`.
- Для маршрутизации используйте `tools_condition` из `langgraph.prebuilt` - он уже умеет определять, нужно ли вызывать инструменты.
- Для визуализации: `graph.get_graph().draw_mermaid_png()`.

Пример скелета графа:
```python
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def call_model(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")
graph = builder.compile()
```


### 2.2 - 2.3 Промпт для Оркестратора и создание субагентов (15 баллов)

**Задание:**
1. Разделите ваши 3 инструмента между двумя субагентами (например, Агент-Аналитик и Агент-Информатор).
2. Напишите системный промпт для Оркестратора, описывающий компетенции каждого субагента и правила маршрутизации.
3. Реализуйте субагентов как отдельные узлы в графе.
4. Оркестратор должен анализировать запрос пользователя и направлять его нужному субагенту.

**Подсказки по промпту Оркестратора:**
- Четко опишите, что умеет каждый субагент. Чем точнее описание - тем лучше маршрутизация.
- Укажите, что делать, если запрос не подходит ни одному субагенту.
- Попросите Оркестратора объяснять свое решение о маршрутизации.

**Подсказки по архитектуре:**
- Каждый субагент - это отдельная функция-узел в графе, которая вызывает своего LLM с набором инструментов.
- Оркестратор может быть реализован как узел с `conditional_edges`, которые смотрят на решение LLM.
- Для передачи контекста между агентами используйте поле `messages` в состоянии графа.
- Можно добавить кастомные поля в состояние (например, `current_agent: str`) для отслеживания маршрута.

Пример структуры мультиагентного графа:
```python
class AgentState(MessagesState):
    current_agent: str  # какой агент сейчас работает

def orchestrator_node(state):
    # LLM решает, кому делегировать
    ...

def analyst_agent_node(state):
    # Субагент с инструментами анализа
    ...

def info_agent_node(state):
    # Субагент с инструментами получения информации
    ...
```


In [ ]:
# TODO: Напишите системный промпт для Оркестратора
orchestrator_prompt = """
Вы - Оркестратор. Ваша задача - принять запрос пользователя и направить его нужному субагенту.

У вас есть два субагента:
1. Агент-Аналитик: умеет [опишите компетенции].
2. Агент-Информатор: умеет [опишите компетенции].

Правила маршрутизации:
- Если запрос требует [условие] - направьте к Агент-Аналитику.
- Если запрос требует [условие] - направьте к Агент-Информатору.
- Если запрос не подходит ни одному - вежливо откажитесь.
"""

# TODO: Определите узлы субагентов

# TODO: Определите узел Оркестратора и логику маршрутизации

# TODO: Соберите мультиагентный граф и визуализируйте его


### 2.4 Human-in-the-loop (Безопасность) (10 баллов)

**Задание:**
1. Добавьте инструмент `send_report_to_management` (может просто печатать текст или сохранять в файл).
2. Настройте граф так, чтобы перед вызовом этого инструмента выполнение приостанавливалось и ожидало ручного подтверждения.

**Подсказки:**
- В LangGraph для паузы используется параметр `interrupt_before=["tools"]` при компиляции графа.
- Для сохранения состояния во время паузы нужен `checkpointer`. Используйте `MemorySaver` для тестирования.
- Каждый запуск графа должен иметь уникальный `thread_id` в `config` - это идентификатор сессии.
- Чтобы возобновить выполнение, вызовите граф повторно с тем же `thread_id` и `None` в качестве входных данных.
- Используйте `graph.get_state(config)` чтобы проверить текущее состояние и убедиться, что граф на паузе.

Пример паузы и возобновления:
```python
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
graph = builder.compile(checkpointer=memory, interrupt_before=["tools"])

config = {"configurable": {"thread_id": "session-1"}}

# Первый запуск - граф остановится перед вызовом инструмента
result = graph.invoke({"messages": [("user", "запрос")]}, config)

# Проверяем состояние
state = graph.get_state(config)
print("Граф на паузе:", state.next)

# Возобновляем выполнение (подтверждение)
final_result = graph.invoke(None, config)
```


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# TODO: Создайте инструмент send_report_to_management
@tool
def send_report_to_management(report_text: str) -> str:
    """Отправляет финальный отчет руководству. Используй только когда пользователь явно просит отправить отчет.
    Args:
        report_text: Текст отчета для отправки.
    """
    # TODO: Ваша реализация (например, сохранить в файл или напечатать)
    pass

# TODO: Добавьте инструмент одному из субагентов

# TODO: Создайте checkpointer и скомпилируйте граф с interrupt_before
# memory = MemorySaver()
# graph_with_hitl = builder.compile(checkpointer=memory, interrupt_before=["tools"])


### 2.5 Финальное тестирование мультиагентной системы (5 баллов)

Продемонстрируйте полный цикл работы вашей мультиагентной системы.

Задайте сложный запрос, который:
1. Требует делегирования от Оркестратора к субагенту.
2. Заканчивается вызовом инструмента `send_report_to_management`.

Покажите все четыре этапа: запуск, пауза перед отправкой, ручное подтверждение, финальный ответ.

**Подсказка:** Выводите промежуточные состояния графа, чтобы было видно, как меняется `state.next` до и после подтверждения.


In [ ]:
# TODO: Этап 1 - Запустите граф с комплексным запросом
config = {"configurable": {"thread_id": "final-test-1"}}
# result = graph_with_hitl.invoke({"messages": [("user", "ваш запрос")]}, config)


In [ ]:
# TODO: Этап 2 - Проверьте, что граф на паузе
# state = graph_with_hitl.get_state(config)
# print("Следующий шаг:", state.next)
# print("Последнее сообщение:", state.values["messages"][-1])


In [ ]:
# TODO: Этап 3 - Дайте подтверждение и возобновите выполнение
# final_result = graph_with_hitl.invoke(None, config)


In [ ]:
# TODO: Этап 4 - Выведите финальный ответ
# print(final_result["messages"][-1].content)


### 2.6 Анализ рисков и идеи по улучшению мультиагентной системы (10 баллов)

Это задание не оценивается в баллах, но является обязательным. Здесь вы должны проявить системное мышление и осмыслить архитектуру, которую построили.

**Задание:** Напишите развернутый анализ (минимум 400 слов) в ячейке ниже, ответив на следующие вопросы:

**Риски мультиагентной архитектуры:**
- Что произойдет, если Оркестратор неправильно определит нужного субагента? Как часто это может происходить и почему?
- Как растет стоимость и задержка при добавлении новых субагентов? Когда мультиагентность становится избыточной?
- Насколько надежен механизм Human-in-the-loop? Что если человек нажмет "подтвердить" не глядя?
- Какие риски несет общее состояние (`messages`) между агентами? Может ли один субагент "запутать" другого?

**Гипотезы по улучшению:**
- Как можно улучшить качество маршрутизации Оркестратора? Например, добавить классификатор намерений (intent classifier) перед Оркестратором.
- Как добавить долгосрочную память агентам? Например, сохранять важные факты из разговоров в векторную базу данных.
- Как реализовать параллельное выполнение субагентов, если запрос требует работы нескольких из них одновременно?
- Как можно автоматически оценивать качество ответов агентов (LLM-as-a-judge)?

**Идеи по расширению:**
- Какие новые субагенты и инструменты сделали бы вашу систему значительно полезнее?
- Как бы вы развернули эту систему в продакшене? Какую инфраструктуру выбрали бы?
- Как реализовать мониторинг и трассировку работы агентов в реальном времени (например, через Arize Phoenix или LangSmith)?
- Как обеспечить безопасность системы от prompt injection атак, когда злоумышленник пытается через пользовательский запрос изменить поведение агента?


**Ваш анализ:**

...

---
**Поздравляем с завершением домашнего задания!**

Вы прошли путь от базового ReAct агента до мультиагентной системы с защитой и человеком в контуре. Это фундамент для построения реальных продакшен-систем на базе LLM.